# CausaLab — Weekdays Manifold Steering (INT8 Quantized LLaMA 3.1 8B)

Runs the `weekdays_8b_quantized_pipeline` experiment on Colab Pro using INT8 BitsAndBytes quantization (~8 GB VRAM).

**Before running:** add the following secrets in *Colab → Secrets*:
- `GITHUB_TOKEN` — a GitHub PAT with `repo` scope
- `GITHUB_USERNAME` — your GitHub username (e.g. `kgreyy`)
- `HF_TOKEN` — a HuggingFace token with access to `meta-llama/Llama-3.1-8B`

**Runtime:** select a GPU runtime (A100 recommended; T4/V100 also work).

## 1. Authenticate & clone

In [ ]:
import os
from google.colab import userdata

os.environ['GITHUB_TOKEN']    = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_USERNAME'] = userdata.get('GITHUB_USERNAME')
os.environ['HF_TOKEN']        = userdata.get('HF_TOKEN')

print('Secrets loaded.')

In [ ]:
!git clone --branch manifold_steering \
    https://$GITHUB_TOKEN@github.com/$GITHUB_USERNAME/causalab.git
%cd causalab

## 2. Install dependencies

In [ ]:
# Install uv (fast package manager used by this project)
!pip install -q uv
!uv --version

In [ ]:
# Install all project deps (including bitsandbytes, pyvene from git, hydra, etc.)
!uv sync --no-dev 2>&1 | tail -5
print('Dependencies installed.')

## 3. Verify GPU & quantization setup

In [ ]:
!uv run python - <<'EOF'
import torch, bitsandbytes as bnb
print(f'PyTorch  : {torch.__version__}')
print(f'bitsandbytes: {bnb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
EOF

## 4. Run the weekdays INT8 pipeline

This runs all six analyses in sequence (baseline → subspace → activation_manifold → output_manifold → path_steering → pullback) on the weekdays task using an INT8-quantized LLaMA 3.1 8B.

In [ ]:
!chmod +x scripts/run_exp.sh
!scripts/run_exp.sh weekdays_8b_quantized_pipeline 2>&1

## 5. View results

Results are written under `artifacts/natural_domains_arithmetic/llama31_8b_int8/`.

In [ ]:
import os, json, pathlib

root = pathlib.Path('artifacts/natural_domains_arithmetic/llama31_8b_int8')
for p in sorted(root.rglob('*.json')):
    print(p)
    try:
        print(json.dumps(json.loads(p.read_text()), indent=2)[:400])
    except Exception:
        pass
    print()

## 6. (Optional) Save artifacts to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/causalab_results/weekdays_int8'
shutil.copytree('artifacts', dest, dirs_exist_ok=True)
print(f'Saved to {dest}')